In [2]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

Loading environment cabt failed: dlopen(/Users/fcp/opt/anaconda3/envs/orbitwars/lib/python3.11/site-packages/kaggle_environments/envs/cabt/cg/libcg.so, 0x0006): tried: '/Users/fcp/opt/anaconda3/envs/orbitwars/lib/python3.11/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (not a mach-o file), '/System/Volumes/Preboot/Cryptexes/OS/Users/fcp/opt/anaconda3/envs/orbitwars/lib/python3.11/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (no such file), '/Users/fcp/opt/anaconda3/envs/orbitwars/lib/python3.11/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (not a mach-o file)
Loading environment open_spiel_env failed: No module named 'pyspiel'
Environment: orbit_wars v1.0.9
Players: [2, 4]
Max steps: 500


In [4]:
#see the observations rq
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

#understanding action space

# Return a list of moves:

# # [[from_planet_id, direction_angle, num_ships], ...]
# # from_planet_id: ID of a planet you own.
# # direction_angle: Angle in radians (0 = right, pi/2 = down).
# # num_ships: Integer number of ships to send.





Timeout: 
Player: 0
Angular velocity: 0.0415 rad/turn

Planets (32):
  id=0 owner=Neutral    pos=(97.0, 78.1) r=2.4 ships=34 prod=4
  id=1 owner=Neutral    pos=(3.0, 78.1) r=2.4 ships=34 prod=4
  id=2 owner=Neutral    pos=(97.0, 21.9) r=2.4 ships=34 prod=4
  id=3 owner=Neutral    pos=(3.0, 21.9) r=2.4 ships=34 prod=4
  id=4 owner=Player 0   pos=(96.8, 63.4) r=2.4 ships=10 prod=4
  id=5 owner=Neutral    pos=(3.2, 63.4) r=2.4 ships=58 prod=4


In [ ]:
#undestanding the game structures
#fleets that cross the sun (50, 50) get destroyed.

#radius: Determined by production: 1 + ln(production). Higher production planets are physically larger.
# production: Integer from 1 to 5. Each turn, an owned planet generates this many ships.
# ships: Current garrison. Starts between 5 and 99 (skewed toward lower values).

# orbital_radius + planet_radius < 50 rotate around the sun at a constant angular velocity (0.025-0.05 radians/turn, randomized per game). Use initial_planets and angular_velocity from the observation to predict their positions.

#fleet speed dynamics

# Fleet Speed
# Fleet speed scales with size on a logarithmic curve:

# speed = 1.0 + (maxSpeed - 1.0) * (log(ships) / log(1000)) ^ 1.5
# 1 ship moves at 1.0 units/turn.
# Larger fleets move faster, approaching the maximum speed (default 6.0).
# A fleet of ~500 ships moves at ~5, and ~1000 ships reaches the max.

# Combat
# When one or more fleets collide with a planet (either by flying into it or being swept by a moving planet), combat is resolved:

# All arriving fleets are grouped by owner. Ships from the same owner are summed.
# The largest attacking force fights the second largest. The difference in ships survives.
# If there is a surviving attacker:
# If the attacker is the same owner as the planet, the surviving ships are added to the garrison.
# If the attacker is a different owner, the surviving ships fight the garrison. If the attackers exceed the garrison, the planet changes ownership and the garrison becomes the surplus.
# If two attackers tie, all attacking ships are destroyed (no survivors).




In [ ]:
#game dynamic notes
#partial observability aspect: no insight into which fleets are flying or their sizes. Only the current state of planets is visible. This adds a layer of uncertainty and strategy, as players must infer potential threats and opportunities based on limited information.

#termination state 

# The game ends when:

# Step limit reached: 500 turns.
# Elimination: Only one player (or zero) remains with any planets or fleets.
# Final score = total ships on owned planets + total ships in owned fleets. Highest score wins.

#think about early-mid-late game strategy
#early game should be focused on expansion - falling behind early could be fatal.
#mid game - this is much more nuanced.
#late game - do we end the game early, or do we play defense until terminataion?

#How do you decide an attack is “high-confidence”?
#Timing and geometry - if you can predict a planet will be undefended for a turn, that’s a good time to strike
#opponent avaialable resources - if you can predict an opponent is low on ships, that’s a good time to strike
#arrival time advantage - if you can strike a planet just as an opponent’s fleet arrives, you can potentially destroy their fleet and take the planet in one move

In [ ]:
#model architecture

#separation of value, feasibility, and decisionmaking

